# Lesson 00 — Run a graph from the notebook & examine every step

The foundational **inspection harness**: run a graph and watch its *intermediate*
state, not just the final answer. We inspect the lesson-04 validation loop, because its
retry cycle makes the intermediaries actually change between steps.

Everything except the clearly-marked "real model" cell uses a **stub** — a fake model
that hallucinates a citation once, then fixes it — so the loop replays deterministically
with **no API cost**. Spec: `specs/00_run_and_inspect.md`.

In [ ]:
from clinical import (
    Evidence,
    SymptomProblem,
    SymptomExtraction,
    show_symptom_evidence_matrix,
)
from graphs.validation_loop import build_graph

# A tiny synthetic transcript so we control which excerpts are verbatim substrings.
TRANSCRIPT = "[00:00] Patsient: valu on parema alakohu piirkonnas. [00:12] Arst: kas kiirgub?"
INPUT = {"transcript": TRANSCRIPT, "encounter_date": "27.11.2025"}

# Two canned extractions: one cites a verbatim excerpt (passes validation), one invents it.
GOOD = SymptomExtraction(problems=[SymptomProblem(
    problem="parema alakohu valu",
    core_evidence=[Evidence(speaker="patient", timestamp="00:00",
                            excerpt="valu on parema alakohu piirkonnas")])])   # verbatim -> passes
BAD = SymptomExtraction(problems=[SymptomProblem(
    problem="parema alakohu valu",
    core_evidence=[Evidence(speaker="patient", timestamp="00:00",
                            excerpt="valu vasakus ola piirkonnas")])])          # never said -> flagged


class BadThenGoodModel:
    """Fake model: hallucinates a citation on call 1, fixes it on call 2.

    Lets us watch the validation loop actually cycle, deterministically and for free.
    (Same stub idea as the lesson-04 tests.)
    """

    def __init__(self):
        self.calls = 0

    def invoke(self, messages):
        self.calls += 1
        return BAD if self.calls == 1 else GOOD


def fresh_graph():
    """A NEW graph + NEW stub each time, so every demo below replays the full loop."""
    return build_graph(model=BadThenGoodModel())


def summarize(delta):
    """Compact one-line view of what a node returned (instead of dumping Pydantic)."""
    out = {}
    for key, value in delta.items():
        if key == "extraction":
            excerpt = value.problems[0].core_evidence[0].excerpt if value.problems else "-"
            out["extraction"] = f"{len(value.problems)} problem(s), excerpt={excerpt!r}"
        elif key == "errors":
            out["errors"] = f"{len(value)} unsupported"
        else:
            out[key] = value
    return out

## 1. `invoke()` — run to completion, see only the final result

The everyday call. You get the final state, but **none** of the steps in between.

In [ ]:
final = fresh_graph().invoke(INPUT)
print("attempts:", final["attempts"], " | unresolved errors:", len(final["errors"]))
show_symptom_evidence_matrix(final["extraction"])

## 2. `stream(stream_mode="updates")` — what each node *changed*

Now the intermediaries. Each item is `{node_name: delta}`. Watch `extract -> validate`
run **twice**: the first extract hallucinates, `validate` flags it, the loop repairs it.

In [ ]:
for step in fresh_graph().stream(INPUT, stream_mode="updates"):
    for node, delta in step.items():
        print(f"[{node}]  {summarize(delta)}")

## 3. `stream(stream_mode="values")` — the full state after each step

Same run, but each item is the **entire** accumulated state. Watch `attempts` climb
`1 -> 2` and `errors` fall `1 -> 0`. (Subtle: right after the 2nd `extract`, `errors` is
still stale until `validate` reruns — only visible in the intermediaries.)

In [ ]:
for state in fresh_graph().stream(INPUT, stream_mode="values"):
    extraction = state.get("extraction")
    print(
        f"attempts={state.get('attempts')}  "
        f"problems={len(extraction.problems) if extraction else 0}  "
        f"errors={len(state.get('errors', []))}"
    )

## 4. `stream(stream_mode="debug")` — task ordering & the loop

The most verbose mode (task starts/results + checkpoints). Good for seeing exactly which
node ran when — here, four tasks: extract, validate, extract, validate.

In [ ]:
for event in fresh_graph().stream(INPUT, stream_mode="debug"):
    if event["type"] in ("task", "task_result"):
        print(f"step {event['step']}: {event['type']:12} -> {event['payload'].get('name', '-')}")

## 5. The real model — one call, opt-in

Everything above is free (stub). This is the **only** cell that hits OpenRouter, and it's
guarded so "Run All" won't surprise-charge you: set `RUN_REAL=1` in the environment (and
have `OPENROUTER_API_KEY` in `.env`) to enable it.

In [ ]:
import os

if os.environ.get("RUN_REAL"):
    from clinical import get_transcript

    result = build_graph().invoke(
        {"transcript": get_transcript("transcript_1"), "encounter_date": "27.11.2025"}
    )
    print("attempts:", result["attempts"], " | unresolved errors:", len(result["errors"]))
    show_symptom_evidence_matrix(result["extraction"])
else:
    print("Skipped the real model. Set RUN_REAL=1 (with OPENROUTER_API_KEY) to run it.")

## Where to examine the *full* trace: LangSmith

Tracing is already on (`LANGSMITH_TRACING=true`, project `agent_01`), so every real run
is recorded at **[smith.langchain.com](https://smith.langchain.com) -> `agent_01`**: the
trace tree, each node's inputs/outputs, and the exact prompts sent to the model (including
the repair message on the retry). That's the deepest view — richer than anything printed
here or shown in Studio.

**Next:** lesson 09 goes deeper on streaming (token-level `messages`, `custom` events);
lesson 06 adds a checkpointer so you can *replay* past steps with `get_state_history`.